# Prompt Optimization: Practice Exercise

In this exercise, you'll implement the **"Tool Usage Over Guessing"** prompt optimization technique to prevent an agent from making up information when tool data is unavailable.

**What you'll implement:**
- A system prompt that prevents hallucination by instructing the agent to never guess or fabricate information

**Estimated time:** 10-15 minutes

## Setup

Run this cell to load dependencies and create the weather tools you'll use.

In [ ]:
# Setup - run this cell first

from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get current weather information for a specific location.
    
    Args:
        location: The city or location to get weather for
    
    Returns:
        A string describing the current weather conditions, or a message if data is not available
    """
    # Mock implementation with limited data
    weather_data = {
        "new york": "Sunny, 72F",
        "london": "Cloudy, 59F",
        "tokyo": "Clear, 68F",
        "paris": "Rainy, 55F",
    }
    
    location_key = location.lower().strip()
    
    if location_key in weather_data:
        return f"Weather in {location}: {weather_data[location_key]}"
    else:
        return f"Location not found"


@tool
def get_forecast(location: str) -> str:
    """Get 3-day weather forecast for a specific location.
    
    Args:
        location: The city or location to get forecast for
    
    Returns:
        A string with the 3-day forecast, or a message if data is not available
    """
    # Mock implementation with limited data
    forecast_data = {
        "new york": "Day 1: Sunny, 72F | Day 2: Partly cloudy, 68F | Day 3: Rainy, 65F",
        "london": "Day 1: Cloudy, 59F | Day 2: Rainy, 57F | Day 3: Foggy, 56F",
        "tokyo": "Day 1: Clear, 68F | Day 2: Sunny, 70F | Day 3: Partly cloudy, 67F",
        "paris": "Day 1: Rainy, 55F | Day 2: Cloudy, 58F | Day 3: Sunny, 62F",
    }
    
    location_key = location.lower().strip()
    
    if location_key in forecast_data:
        return f"3-day forecast for {location}: {forecast_data[location_key]}"
    else:
        return f"Forecast data not available for {location}"


tools = [get_weather, get_forecast]


# Configure the language model
model = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1
)


def ask_agent_with_streaming(agent, question: str):
    """Ask the agent a question and stream the response to show reasoning.
    
    Args:
        agent: The LangChain agent to query
        question: The question to ask
    """
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}\n")
    
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="updates"
    ):
        if "model" in chunk:
            messages = chunk["model"].get("messages", [])
            for msg in messages:
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tool_call in msg.tool_calls:
                        print(f"Tool Call: {tool_call['name']}")
                        print(f"   Input: {tool_call['args']}")
                elif hasattr(msg, 'content') and msg.content:
                    print(f"\nAgent Response:\n{msg.content}")
        
        if "tools" in chunk:
            messages = chunk["tools"].get("messages", [])
            for msg in messages:
                if hasattr(msg, 'content'):
                    print(f"   Result: {msg.content}\n")
    
    print(f"\n{'='*60}\n")


print("Setup complete!")
print(f"Available tools: {[t.name for t in tools]}")

## Context

You are building a weather assistant that helps users get weather information. The agent has access to two tools:

- `get_weather(location)` - Returns current weather for known locations (New York, London, Tokyo, Paris)
- `get_forecast(location)` - Returns 3-day forecast for known locations

**The problem:** Without proper prompt optimization, the agent may make up weather information for unknown locations instead of admitting it lacks data.

Your task is to write an optimized system prompt that guides the agent to:
- Never guess or fabricate weather information
- Clearly state when data is not available
- Only report information directly from tools

## The Problem: Demonstrating Hallucination

First, let's see what happens with a basic agent when we ask about an unknown location. The agent might try to guess or provide general information instead of admitting it doesn't have data.

In [ ]:
# Create a basic agent without anti-hallucination instructions
basic_prompt = """You are a helpful weather assistant.

Use the available tools to help users with weather questions.
"""

basic_agent = create_agent(
    model=model.with_config(configurable={"system_prompt": basic_prompt}),
    tools=tools
)

print("Basic agent created!")

In [ ]:
# Test with unknown location - watch for potential hallucination
ask_agent_with_streaming(
    basic_agent,
    "What's the weather like in Mumbai?"
)

## Your Task: Implement "Tool Usage Over Guessing"

Write a system prompt that prevents the agent from making up information. Your prompt should:

1. **Explicitly forbid guessing** - Tell the agent to NEVER guess, estimate, or make up weather information
2. **Handle "not available" responses** - When a tool returns "not found" or "not available", the agent should clearly tell the user it doesn't have data for that location
3. **Prohibit general information** - Don't let the agent provide general climate info or typical weather patterns as a fallback
4. **Require tool verification** - Only report information that was directly retrieved from tools

In [ ]:
def create_anti_hallucination_prompt() -> str:
    """Create a system prompt with Tool Usage Over Guessing optimization.
    
    The prompt should instruct the agent to:
    - Never guess or make up weather information
    - Clearly state when data is not available
    - Not provide general climate information as a fallback
    - Only report information directly from tools
    
    Returns:
        A system prompt string with anti-hallucination instructions
    """
    # TODO: Write your system prompt with Tool Usage Over Guessing optimization
    # Include:
    # - Agent role description (weather assistant)
    # - Explicit instructions to NEVER guess or fabricate data
    # - Instructions for handling "not available" responses from tools
    # - Prohibition against general/typical weather information
    pass

## Test Your Implementation

Run these tests to verify your prompt prevents hallucination.

In [ ]:
# Create agent with your anti-hallucination prompt
anti_hallucination_prompt = create_anti_hallucination_prompt()

optimized_agent = create_agent(
    model=model.with_config(configurable={"system_prompt": anti_hallucination_prompt}),
    tools=tools
)

print("Optimized agent created!")
print(f"\nYour prompt:\n{anti_hallucination_prompt}")

In [ ]:
# Test 1: Unknown location - agent should NOT make up weather
ask_agent_with_streaming(
    optimized_agent,
    "What's the weather like in Mumbai?"
)

In [ ]:
# Test 2: Another unknown location
ask_agent_with_streaming(
    optimized_agent,
    "What's the forecast for Sydney this week?"
)

In [ ]:
# Test 3: Verify known locations still work correctly
ask_agent_with_streaming(
    optimized_agent,
    "What's the weather in Tokyo?"
)

## Verify Your Solution

Your implementation is successful if:

**For unknown locations (Mumbai, Sydney):**
- Agent uses the appropriate tool to check for data
- Agent clearly states data is not available
- Agent does NOT provide made-up weather conditions
- Agent does NOT provide general information like "Mumbai typically has hot weather..."

**For known locations (Tokyo, London, Paris, New York):**
- Agent returns correct data from tools
- Agent continues to function normally